In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
# load and explore the hypothetical prediction data; the data assumes all female passengers survived while all male passengers died.
input_data = pd.read_csv('/kaggle/input/competitions/titanic/gender_submission.csv')
input_data.head(10)

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0


In [3]:
# load and explore the training data
train_data = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
train_data.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [4]:
# check the percentage of women who survived in the training data
women = train_data.loc[train_data.Sex == 'female']['Survived']
women_percent = round((sum(women)/len(women)) * 100, 1)

# check the percentage of men who survived in the training data
men = train_data.loc[train_data.Sex == 'male']['Survived']
men_percent = round((sum(men)/len(men)) * 100, 1)

# print the percentages of the survivors
print(f'{women_percent}% of women survived in the training data')

print(f'{men_percent}% of men survived in the training data')

74.2% of women survived in the training data
18.9% of men survived in the training data


In [5]:
# load and explore the testing data
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [6]:
# split the training data to X and Y variables
x_features = ['Pclass','Sex','Age','SibSp','Parch']
X_train = pd.get_dummies(train_data[x_features])
Y_train = train_data['Survived']

# identify the feature variables in the test data
X_test = pd.get_dummies(test_data[x_features])

X_train.shape, X_test.shape

((891, 6), (418, 6))

In [7]:
# initialize the xgboost model
model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=7,
    eval_metric='logloss',
    random_state=117
)

# fit the model
model.fit(X_train, Y_train, verbose=True)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=7, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [8]:
# make predictions on the test data
Y_pred = model.predict(X_test)

# create a table of the predictions and the test data
pred_table = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Sex': test_data.Sex, 'Survived': Y_pred})
pred_table.head(10)

,PassengerId,Sex,Survived
0,892,male,0
1,893,female,0
2,894,male,0
3,895,male,1
4,896,female,1
5,897,male,0
6,898,female,0
7,899,male,0
8,900,female,0
9,901,male,0


In [9]:
# check the percentage of men who survived in predictions
men_pred = pred_table.loc[pred_table.Sex == 'male']['Survived']
men_percent_pred = (sum(men_pred)/len(men_pred))*100

# check the percentage of women who survived in the predictions
women_pred = pred_table.loc[pred_table.Sex == 'female']['Survived']
women_percent_pred = (sum(women_pred)/len(women_pred))

# print the percentages of men and women survivors in the predictions for comparison with the train data and the assumed hypothetical
print(f'{women_percent_pred:.1f}% of women survived in the prediction')

print(f'{men_percent_pred:.1f}% of men survived in the prediction')

0.8% of women survived in the prediction
12.4% of men survived in the prediction


In [10]:
# initialize a RandomForestClassifier model
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=117
)

#fit the RFC model
rf_model.fit(X_train, Y_train)

RandomForestClassifier(n_estimators=200, random_state=117)

In [11]:
# make predictions on the random forest model
rf_pred = rf_model.predict(X_test)

# create a predictions and test data table from the rf model predictions
rf_pred_table = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Sex': test_data.Sex, 'Survived': rf_pred})
rf_pred_table.head(10)

,PassengerId,Sex,Survived
0,892,male,0
1,893,female,0
2,894,male,1
3,895,male,1
4,896,female,0
5,897,male,0
6,898,female,0
7,899,male,0
8,900,female,1
9,901,male,0


In [12]:
# check the percentage of men who survived in rf_model predictions
rf_men_pred = rf_pred_table.loc[rf_pred_table.Sex == 'male']['Survived']
men_percent_rf = (sum(rf_men_pred)/len(rf_men_pred))*100

# check the percentage of women who survived in the rf_model predictions
rf_women_pred = rf_pred_table.loc[rf_pred_table.Sex == 'female']['Survived']
women_percent_rf = (sum(rf_women_pred)/len(rf_women_pred))

# print the percentages of men and women survivors in the predictions for comparison with the train data and the assumed hypothetical
print(f'{women_percent_rf:.1f}% of women survived in the random forest model prediction')

print(f'{men_percent_rf:.1f}% of men survived in the random forest model prediction')

0.8% of women survived in the random forest model prediction
14.3% of men survived in the random forest model prediction


In [13]:
# initialize a stacking classifier with the previous models as the base estimators
estimators = [
    ('rf', rf_model),
    ('xgb', model)
]

stacking_model = StackingClassifier(
    estimators,
    final_estimator=LogisticRegression(),
    cv="prefit"
)

# fit the model
stacking_model.fit(X_train, Y_train)

StackingClassifier(cv='prefit',
                   estimators=[('rf',
                                RandomForestClassifier(n_estimators=200,
                                                       random_state=117)),
                               ('xgb',
                                XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=None,
                                              device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='logloss',
                                              feature_types=None,
                                              featu...
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=7,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=200, n_jobs=None,
                                              num_parallel_tree=None, ...))],
                   final_estimator=LogisticRegression())

In [14]:
# make predictions on the stacking model
sm_pred = stacking_model.predict(X_test)

# create a predictions and test data table from the stacking model predictions
sm_pred_table = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Sex': test_data.Sex, 'Survived': sm_pred})
sm_pred_table.head(10)

,PassengerId,Sex,Survived
0,892,male,0
1,893,female,0
2,894,male,1
3,895,male,1
4,896,female,0
5,897,male,0
6,898,female,0
7,899,male,0
8,900,female,1
9,901,male,0


In [15]:
# check the percentage of men who survived in stacking model predictions
sm_men_pred = sm_pred_table.loc[sm_pred_table.Sex == 'male']['Survived']
men_percent_sm = (sum(sm_men_pred)/len(sm_men_pred))*100

# check the percentage of women who survived in the stacking model predictions
sm_women_pred = sm_pred_table.loc[sm_pred_table.Sex == 'female']['Survived']
women_percent_sm = (sum(sm_women_pred)/len(sm_women_pred))

# print the percentages of men and women survivors in the stacking model predictions for comparison with the train data and the assumed hypothetical
print(f'{women_percent_sm:.1f}% of women survived in the stacking model prediction')

print(f'{men_percent_sm:.1f}% of men survived in the stacking model prediction')

0.8% of women survived in the stacking model prediction
16.2% of men survived in the stacking model prediction


In [16]:
# generate the final output from the stacking model classifer and output columns "PassengerId", and 'Survived'.
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': sm_pred})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
